# 🚁 Guardia AI v2.0.1 — Model Export & Testing

**Tackle Studio — Aryan Bajpai**

This notebook:
1. Downloads YOLOv8-nano pretrained weights
2. Exports to TFLite (optimized for Raspberry Pi 4B)
3. Tests inference speed and accuracy
4. Exports person Re-ID model for tailing mode

Run this on **Google Colab (free GPU)** and download the `.tflite` files to the Pi.

In [ ]:
# Install dependencies
!pip install ultralytics tflite-support onnx2tf tensorflow-lite-support -q

In [ ]:
from ultralytics import YOLO
import time

# Load YOLOv8-nano (smallest, fastest — perfect for Pi 4B)
model = YOLO('yolov8n.pt')

print(f'Model: {model.info()}')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()) / 1e6:.1f}M')

## Export to TFLite (INT8 quantized for Pi 4B)

In [ ]:
# Export to TFLite with INT8 quantization
# INT8 reduces model size ~4x and speeds up CPU inference on Pi
model.export(
    format='tflite',
    imgsz=320,      # Small input for speed
    int8=True,       # INT8 quantization
    half=False,      # Pi 4B doesn't support FP16 well
)

print('\n✅ TFLite export complete!')
print('File: yolov8n_saved_model/yolov8n_integer_quant.tflite')

In [ ]:
# Also export FP32 version (more accurate, slightly slower)
model.export(
    format='tflite',
    imgsz=320,
    int8=False,
)

print('\n✅ FP32 TFLite export complete!')

## Test Inference Speed

In [ ]:
import numpy as np

# Simulate Pi 4B inference (Colab has faster CPU, so times will be lower)
# On Pi 4B, expect ~80-120ms per frame at 320x320

# Test with TFLite
tflite_model = YOLO('yolov8n_saved_model/yolov8n_float32.tflite')

# Warm up
dummy = np.random.randint(0, 255, (320, 320, 3), dtype=np.uint8)
for _ in range(5):
    tflite_model.predict(dummy, verbose=False)

# Benchmark
times = []
for _ in range(50):
    start = time.time()
    tflite_model.predict(dummy, verbose=False)
    times.append((time.time() - start) * 1000)

print(f'Inference time (Colab CPU):')
print(f'  Mean:   {np.mean(times):.1f} ms')
print(f'  Median: {np.median(times):.1f} ms')
print(f'  Min:    {np.min(times):.1f} ms')
print(f'  Max:    {np.max(times):.1f} ms')
print(f'  FPS:    {1000 / np.mean(times):.1f}')
print()
print('Note: Pi 4B will be ~3-5x slower than Colab CPU.')
print('Expected Pi 4B: 80-120ms/frame → 8-12 FPS')

## Test Detection Quality

In [ ]:
# Test on sample images
results = model.predict(
    source='https://ultralytics.com/images/bus.jpg',
    conf=0.35,
    iou=0.45,
    imgsz=320,
    save=True,
)

for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        name = r.names[cls]
        print(f'  {name}: {conf:.2f}')

## Custom Training (optional)
Fine-tune on your specific surveillance/rescue dataset.

In [ ]:
# Uncomment to fine-tune on a custom dataset
# model.train(
#     data='path/to/your/data.yaml',
#     epochs=50,
#     imgsz=320,
#     batch=16,
#     device='0',  # GPU
#     name='guardia_yolov8n',
# )

print('To fine-tune, prepare a YOLO-format dataset with these classes:')
print('  - person (primary target)')
print('  - vehicle, bicycle, motorcycle')
print('  - backpack, suitcase (for rescue scenarios)')
print('  - dog, cat (for animal rescue)')

## Download Models
Download the exported `.tflite` files and copy them to:
```
v2.0.1/drone/ai/models/yolov8n.tflite
```

In [ ]:
# Package for download
import shutil
import os

os.makedirs('guardia_models', exist_ok=True)

# Copy TFLite models
for f in ['yolov8n_saved_model/yolov8n_float32.tflite',
          'yolov8n_saved_model/yolov8n_integer_quant.tflite']:
    if os.path.exists(f):
        shutil.copy(f, 'guardia_models/')
        print(f'Copied: {f}')

# Also copy the original PyTorch weights
shutil.copy('yolov8n.pt', 'guardia_models/')

# Zip for download
shutil.make_archive('guardia_models', 'zip', '.', 'guardia_models')
print('\n✅ Download: guardia_models.zip')

# If on Colab, trigger download
try:
    from google.colab import files
    files.download('guardia_models.zip')
except ImportError:
    print('Not on Colab — manually download guardia_models.zip')

## Memory Budget Check
Verify everything fits in 8GB RAM on Pi 4B.

In [ ]:
import os

print('📊 Memory Budget for Raspberry Pi 4B (8GB)')
print('=' * 50)

budget = {
    'OS + System': 500,
    'Python runtime': 100,
    'YOLOv8-nano TFLite (INT8)': 8,
    'YOLOv8-nano TFLite (FP32)': 25,
    'Inference buffers (320x320)': 50,
    'Camera buffers (720p dual)': 150,
    'Video encoder (H.264)': 100,
    'MAVLink + pymavlink': 30,
    'ByteTracker': 20,
    'FastAPI (if GCS on same Pi)': 200,
    'Recording buffer': 100,
    'Overhead & safety margin': 500,
}

total = sum(budget.values())
available = 8192

for name, mb in budget.items():
    bar = '█' * (mb // 20)
    print(f'  {name:35s} {mb:5d} MB  {bar}')

print(f'\n  {"TOTAL":35s} {total:5d} MB')
print(f'  {"Available":35s} {available:5d} MB')
print(f'  {"Remaining":35s} {available - total:5d} MB')
print(f'\n{"✅ FITS" if total < available else "❌ OVER BUDGET"}')